[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/colab-slm-playground/blob/main/trending_onnx_chatbot_colab.ipynb)

# Trending ONNX Chatbot

Fetches trending `text-generation` ONNX models from Hugging Face, lets you pick one, and runs a chat UI entirely inside Colab.

Uses [`optimum[onnxruntime]`](https://huggingface.co/docs/optimum/onnxruntime/overview) for loading — no manual session management.

## 1 — Install dependencies

In [ ]:
!pip install -q "optimum[onnxruntime]" transformers huggingface_hub numpy
!wget -q --no-cache https://raw.githubusercontent.com/jonasneves/colab-slm-playground/main/chat_ui.py -O /content/chat_ui.py

## 2 — Fetch trending models

In [ ]:
from huggingface_hub import HfApi
from chat_ui import build_model_table_html, register_load_callback
from IPython.display import display, HTML
from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer, PreTrainedTokenizerFast

model = None
tokenizer = None
loaded_model_id = None

print("Fetching trending ONNX text-generation models...")

api = HfApi()
raw_models = list(api.list_models(
    library="onnx",
    pipeline_tag="text-generation",
    sort="trending_score",
    direction=-1,
    limit=30,
    full=True,
))

model_info = []
for m in raw_models:
    has_onnx = any(
        s.rfilename.endswith(".onnx")
        for s in (m.siblings or [])
    )
    if not has_onnx:
        continue
    model_info.append({
        "id": m.id,
        "downloads": m.downloads or 0,
        "likes": m.likes or 0,
        "gated": getattr(m, "gated", False),
    })

_TOKENIZER_CLASS_ERR = ("does not exist", "not currently imported")

def _load_tokenizer(model_id):
    try:
        return AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    except Exception as e:
        if any(msg in str(e) for msg in _TOKENIZER_CLASS_ERR):
            return PreTrainedTokenizerFast.from_pretrained(model_id)
        raise

_COMPAT_ERRORS = {
    "'list' object has no attribute 'keys'": (
        "Model has a non-standard ONNX config that ORTModelForCausalLM can't parse.\n"
        "If you're trying to load LiquidAI/LFM2.5-350M-ONNX, use lfm2_chatbot_colab.ipynb instead."
    ),
    "Could not find any ONNX": (
        "No standard ONNX file found. The model may use a custom layout incompatible with Optimum.\n"
        "Try a different model from the list."
    ),
}

def _friendly_error(e):
    msg = str(e)
    for pattern, hint in _COMPAT_ERRORS.items():
        if pattern in msg:
            return hint
    return f"Error: {msg}"

def load_model(model_id, variant=""):
    global model, tokenizer, loaded_model_id
    tokenizer = _load_tokenizer(model_id)
    model = ORTModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    loaded_model_id = model_id

register_load_callback(load_model)
display(HTML(build_model_table_html(model_info)))
print(f"Found {len(model_info)} models. Click Load to load a model.")

## 3 — Launch Chat UI

Click **Load** in the table above. Once loaded, run this cell.

In [ ]:
from chat_ui import build_chat_html, register_callback
from IPython.display import display, HTML

assert model is not None, "Click Load in the table above first."

def generate(messages: list) -> str:
    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        parts = [
            f"{'User' if m['role'] == 'user' else 'Assistant'}: {m['content']}"
            for m in messages
        ]
        prompt = "\n".join(parts) + "\nAssistant:"

    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    in_len = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(output_ids[0][in_len:], skip_special_tokens=True).strip()

register_callback(generate)
short_name = loaded_model_id.split("/")[-1]
display(HTML(build_chat_html(short_name, "ONNX")))
print("Chat ready. Each response may take 10–60 s depending on model size.")

---
### Notes

**What is ONNX and why run it on CPU?**
ONNX (Open Neural Network Exchange) is a portable model format that separates the model weights from any specific ML framework. Once a model is exported to ONNX, it can run via ONNX Runtime without PyTorch installed — which means lighter dependencies, faster cold starts, and CPU inference that's often faster than PyTorch on CPU thanks to runtime-level graph optimizations.

**What is Optimum?**
[Optimum](https://huggingface.co/docs/optimum) is Hugging Face's library for exporting and running transformer models on accelerated runtimes. `ORTModelForCausalLM` wraps an ONNX session to expose the same `.generate()` interface as a standard `transformers` model. This is why models exported outside Optimum's convention (like LFM2) don't work here — the KV-cache tensor naming doesn't match what Optimum expects.

**fp32 vs fp16 vs int8 variants**
Most ONNX repos ship multiple precision variants. fp32 is the full-precision baseline. fp16 halves memory usage with negligible quality loss. int8 quantizes weights to 8-bit integers — roughly 4× smaller than fp32, with a small quality tradeoff. `from_pretrained` picks the default (usually fp32); pass `file_name="onnx/model_int8.onnx"` to force a specific one.

**Chat templates**
Instruction-tuned models expect their input formatted in a specific way (e.g. `<|user|>` tags, `[INST]` markers). The tokenizer's `chat_template` field encodes this. Models without one fall back to a plain `User: / Assistant:` prompt, which works but may produce lower-quality responses for models that were fine-tuned on a specific format.